# ConvNeXtTiny — Flag images unlike a reference collection

Use case: flag unusual images in an incoming image stream for review. ConvNeXt is a modern CNN. We define airplane images as the reference category and test airplane-versus-ship novelty using feature distances.

Run the cells from top to bottom. Each notebook is independent.


## Step 1 — Install packages
Use Python 3.11 or 3.12 as a straightforward training environment, or Google Colab. Install into the active notebook kernel. Restart the kernel after the first installation if required. Internet is needed initially for packages, sample images/data and ImageNet weights. A GPU is optional; CPU runs can be slower. Model downloads can be hundreds of MB.


In [ ]:
%pip install tensorflow numpy matplotlib scikit-learn pillow


## Step 2 — Import libraries
TensorFlow runs the neural network. Keras provides the model API. NumPy handles arrays and Matplotlib displays pictures.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)


## Step 3 — Load reference and incoming images
This is category novelty detection, not industrial defect detection. Novelty is always relative to the chosen reference data.


In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
train_labels = train_labels.ravel()
test_labels = test_labels.ravel()
class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
print("Training images:", train_images.shape)
print("Test images:", test_images.shape)
plt.figure(figsize=(9, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(train_images[i])
    plt.title(class_names[train_labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()


## Step 4 — Create disjoint reference, calibration and test samples
Reference images define normal features. Held-out normal calibration images set the threshold. The final test contains new airplane and ship images.


In [ ]:
normal_indices = np.where(train_labels == 0)[0]
reference_images = train_images[normal_indices[:80]]
calibration_images = train_images[normal_indices[80:120]]
normal_test = test_images[np.where(test_labels == 0)[0][:40]]
novel_test = test_images[np.where(test_labels == 8)[0][:40]]
incoming_images = np.concatenate([normal_test, novel_test])
y_true = np.concatenate([np.zeros(40, dtype=int), np.ones(40, dtype=int)])


## Step 5 — Extract normalized ConvNeXt vectors
ConvNeXt includes preprocessing. Feed RGB floats in the 0–255 range. Tiny denotes the smaller variant in this family, but it can still be computationally expensive.


In [ ]:
extractor = keras.applications.ConvNeXtTiny(
    weights="imagenet", include_top=False, pooling="avg", input_shape=(224, 224, 3))

def extract_vectors(images):
    parts = []
    for start in range(0, len(images), 8):
        batch = tf.image.resize(images[start:start + 8], (224, 224))
        parts.append(extractor.predict(batch, verbose=0))
    vectors = np.concatenate(parts)
    return vectors / np.maximum(np.linalg.norm(vectors, axis=1, keepdims=True), 1e-8)

reference_vectors = extract_vectors(reference_images)
calibration_vectors = extract_vectors(calibration_images)
incoming_vectors = extract_vectors(incoming_images)


## Step 6 — Set a threshold using normal calibration images
The novelty score is cosine distance to the nearest reference image. Choose the 95th percentile of held-out normal scores; this targets approximately 5% flagged normals on calibration, not necessarily on future data.


In [ ]:
def novelty_scores(vectors):
    similarities = vectors @ reference_vectors.T
    return 1.0 - np.max(similarities, axis=1)

calibration_scores = novelty_scores(calibration_vectors)
threshold = float(np.quantile(calibration_scores, 0.95))
scores = novelty_scores(incoming_vectors)
predicted_novel = (scores > threshold).astype(int)
print("Novelty threshold:", threshold)


## Step 7 — Evaluate on separate incoming images
Precision measures how many flagged pictures are truly novel. Recall measures how many novel pictures are flagged. Both depend on this demonstration’s airplane/ship definition.


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
print(classification_report(y_true, predicted_novel,
      target_names=["Reference category", "Novel category"], zero_division=0))
print("ROC AUC:", roc_auc_score(y_true, scores))
plt.hist(scores[y_true == 0], bins=12, alpha=0.6, label="Airplane")
plt.hist(scores[y_true == 1], bins=12, alpha=0.6, label="Ship")
plt.axvline(threshold, color="red", linestyle="--", label="Threshold")
plt.xlabel("Novelty score")
plt.legend()
plt.show()


## Step 8 — Inspect the most unusual images
A high score flags dissimilarity, not a guaranteed problem. Human inspection helps explain false positives.


In [ ]:
highest = np.argsort(scores)[::-1][:6]
plt.figure(figsize=(12, 3))
for position, index in enumerate(highest, start=1):
    plt.subplot(1, 6, position)
    plt.imshow(incoming_images[index])
    plt.title(f"{'Ship' if y_true[index] else 'Airplane'}\nScore: {scores[index]:.3f}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## Understand the result
This method learns neither pixel-level defects nor their locations. A changed background can increase novelty. Production use requires representative normal images, meaningful novel examples and calibration on deployment data.


## Practice
1. Change the input image or selected image index.
2. Explain the output in your own words.
3. Compare the result with the original image and record one limitation.


## References
- [Keras Applications](https://keras.io/api/applications/)
- [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html)
- [Keras Grad-CAM example](https://keras.io/examples/vision/grad_cam/)
